# KAN-24 - Hipoteses Estatisticas Antes dos Testes

**Projeto:** Spaceship Titanic - Analise Estatistica  
**Responsavel:** Daniela  
**Revisora dedicada:** Renatta  
**Base analitica:** `data/processed/spaceship_titanic_clean.csv`

## Objetivo desta etapa

Antes de executar qualquer teste inferencial, registramos as perguntas, as hipoteses e a regra de decisao. Esta ordem e importante: se uma hipotese fosse escolhida depois de observar os resultados, poderiamos favorecer explicacoes que parecem significativas apenas por acaso. Essa pratica e conhecida como *p-hacking*.

Esta entrega **nao calcula testes, valores-p, intervalos de confianca, tamanhos de efeito ou conclusoes**. Esses resultados pertencem as tarefas KAN-25, KAN-26 e KAN-27.

## 1. Preparacao reproduzivel

Usaremos apenas a base de treinamento tratada no KAN-12. Ela contem o desfecho observado `Transported` e as variaveis derivadas, incluindo `TotalSpend`. A base `test.csv` do Kaggle nao possui o desfecho e, por isso, nao serve para estes testes de hipoteses.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Permite executar o notebook a partir da raiz do projeto ou da pasta notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'spaceship_titanic_clean.csv'
alpha = 0.05

df = pd.read_csv(DATA_PATH)

print(f'Base carregada: {DATA_PATH}')
print(f'Dimensao: {df.shape[0]:,} passageiros x {df.shape[1]} colunas')
print(f'Nivel de significancia pre-especificado: alpha = {alpha:.2f}')

Base carregada: C:\dev\PosGraduacao\Estatistica\Trabalho Final\spaceship-titanic-estatistica-posdl\data\processed\spaceship_titanic_clean.csv
Dimensao: 8,693 passageiros x 22 colunas
Nivel de significancia pre-especificado: alpha = 0.05


## 2. Contrato dos dados para a inferencia

Antes de formular testes, confirmamos que as variaveis previstas existem, nao possuem ausencias e apresentam as categorias esperadas. Esta verificacao nao e um teste de hipotese: e uma protecao contra analisar a base errada ou uma versao incompleta dela.

In [2]:
REQUIRED_COLUMNS = {'HomePlanet', 'CryoSleep', 'TotalSpend', 'Transported'}
EXPECTED_PLANETS = {'Earth', 'Europa', 'Mars'}

missing_columns = REQUIRED_COLUMNS - set(df.columns)
if missing_columns:
    raise ValueError(f'Colunas obrigatorias ausentes: {sorted(missing_columns)}')

null_counts = df[list(REQUIRED_COLUMNS)].isna().sum()
if (null_counts > 0).any():
    raise ValueError(f'Valores ausentes nas variaveis analiticas: {null_counts[null_counts > 0].to_dict()}')

transported_categories = set(df['Transported'].unique())
if len(transported_categories) != 2:
    raise ValueError(f'Transported deve ter duas categorias; encontrado: {transported_categories}')

observed_planets = set(df['HomePlanet'].unique())
if observed_planets != EXPECTED_PLANETS:
    raise ValueError(
        f'Categorias inesperadas em HomePlanet. Esperado: {EXPECTED_PLANETS}; observado: {observed_planets}'
    )

print('Contrato validado: as quatro variaveis estao completas e disponiveis.')
print('\nContagem por HomePlanet:')
display(df['HomePlanet'].value_counts().rename_axis('HomePlanet').to_frame('n'))
print('\nContagem por CryoSleep:')
display(df['CryoSleep'].value_counts().rename_axis('CryoSleep').to_frame('n'))
print('\nContagem por Transported:')
display(df['Transported'].value_counts().rename_axis('Transported').to_frame('n'))

Contrato validado: as quatro variaveis estao completas e disponiveis.

Contagem por HomePlanet:


,n
HomePlanet,
Earth,4715
Europa,2174
Mars,1804



Contagem por CryoSleep:


,n
CryoSleep,
False,5558
True,3135



Contagem por Transported:


,n
Transported,
True,4378
False,4315


## 3. Conceitos essenciais

Uma **hipotese nula** ($H_0$) representa a ausencia de associacao ou diferenca. A **hipotese alternativa** ($H_1$) representa uma associacao ou diferenca que desejamos investigar.

O nivel de significancia foi fixado em $\alpha = 0{,}05$. Isso limita a 5% o risco planejado de rejeitar $H_0$ quando ela e verdadeira, o chamado **erro do tipo I**. Depois que os testes forem executados, a regra sera:

- se $p < 0{,}05$, rejeitar $H_0$;
- se $p \geq 0{,}05$, nao rejeitar $H_0$.

O valor-p nao e a probabilidade de $H_0$ ser verdadeira. Ele mede quao compativeis os dados observados seriam com um modelo em que $H_0$ fosse verdadeira. Por isso, **nao rejeitar** $H_0$ nao prova igualdade entre os grupos: significa apenas que os dados nao forneceram evidencia suficiente contra ela sob este criterio.

## 4. Pergunta 1 - Planeta de origem e teletransporte

**Pergunta de negocio:** a proporcao de passageiros teletransportados varia conforme o planeta de origem?

- **Variaveis:** `HomePlanet` (Earth, Europa, Mars) e `Transported` (True, False).
- **Unidade de analise:** um passageiro.
- **$H_0$:** `HomePlanet` e `Transported` sao independentes; a distribuicao de `Transported` e a mesma nos planetas de origem.
- **$H_1$:** existe associacao entre `HomePlanet` e `Transported`; pelo menos uma distribuicao/proporcao difere.
- **Teste previsto:** qui-quadrado de independencia, na KAN-25.

O qui-quadrado e adequado porque ambas as variaveis sao categoricas. Ele compara as frequencias observadas em uma tabela de contingencia com as frequencias que seriam esperadas se nao houvesse associacao.

## 5. Pergunta 2 - Criossomo e teletransporte

**Pergunta de negocio:** a proporcao de passageiros teletransportados varia entre quem estava ou nao em criossomo?

- **Variaveis:** `CryoSleep` (True, False) e `Transported` (True, False).
- **Unidade de analise:** um passageiro.
- **$H_0$:** `CryoSleep` e `Transported` sao independentes; a distribuicao de `Transported` e a mesma nos dois estados de criossomo.
- **$H_1$:** existe associacao entre `CryoSleep` e `Transported`; as distribuicoes/proporcoes diferem.
- **Teste previsto:** qui-quadrado de independencia, na KAN-25.

Tambem temos duas variaveis categoricas. O teste respondera se a diferenca observada nas frequencias e grande demais para ser atribuida apenas a variacao amostral sob a hipotese de independencia.

## 6. Pergunta 3 - Gasto total e teletransporte

**Pergunta de negocio:** passageiros teletransportados e nao teletransportados apresentam diferenca no gasto total a bordo?

- **Variaveis:** `TotalSpend` (quantitativa continua/discreta truncada em zero) e `Transported` (categorica binaria: True vs False).
- **Unidade de analise:** um passageiro.
- **Objetivo analitico:** comparar o comportamento de consumo entre quem foi e nao foi teletransportado. Sob a otica economica e financeira do negocio, o foco primario esta na diferenca de volume medio gasto ($\mu$), enquanto a otica distributiva analisa a tendencia central e ordenacao dos postos frente a forte assimetria e presenca massiva de zeros.

**Hierarquia de testes e hipoteses pre-registradas:**

1. **Teste Principal (orientador da conclusao de negocio): Teste t de Welch**
   - **Foco:** diferenca entre as medias populacionais ($\mu_1 - \mu_2$).
   - **$H_0$ (nula):** $\mu_{\text{Transported=True}} = \mu_{\text{Transported=False}}$ (o gasto medio a bordo e igual entre passageiros teletransportados e nao teletransportados).
   - **$H_1$ (alternativa):** $\mu_{\text{Transported=True}} \neq \mu_{\text{Transported=False}}$ (o gasto medio a bordo difere entre os dois grupos).
   - **Justificativa metodologica:** com $N = 8.693$ observacoes (mais de 4.300 por grupo), o Teorema Central do Limite (TCL) assegura a normalidade assintotica da distribuicao amostral das medias. A versao de Welch (`scipy.stats.ttest_ind(..., equal_var=False)`) e rigorosamente apropriada por nao exigir homocedasticidade (igualdade de variancias), acomodando eventuais dispersoes desiguais apontadas pelo Teste de Levene.

2. **Teste Complementar (robustez nao parametrica de postos): Teste Mann-Whitney U**
   - **Foco:** dominancia estocastica e diferenca na distribuicao de postos / medianas.
   - **$H_0$ (nula):** $P(X_{\text{True}} > Y_{\text{False}}) = P(Y_{\text{False}} > X_{\text{True}})$ (as distribuicoes de gastos nos dois grupos sao estocasticamente iguais).
   - **$H_1$ (alternativa):** $P(X_{\text{True}} > Y_{\text{False}}) \neq P(Y_{\text{False}} > X_{\text{True}})$ (uma distribuicao tende a apresentar postos sistematicamente maiores que a outra).
   - **Justificativa metodologica:** `TotalSpend` possui concentracao acentuada de valores nulos (passageiros que nao consumiram) e assimetria positiva pronunciada. O teste de Mann-Whitney (`scipy.stats.mannwhitneyu`) atua como checagem de robustez ordinal para confirmar se a conclusao da media se mantem no nivel dos postos ou se decorre apenas da cauda extrema de altos gastadores.

- **Medidas complementares de efeito e precisao (na KAN-26):**
  - Diferenca absoluta de medias ($\bar{x}_1 - \bar{x}_2$) acompanhada de Intervalo de Confianca de 95% via Welch-Satterthwaite.
  - Tamanho do efeito: $d$ de Cohen para a diferenca parametrica e $r$ de Rosenthal ($r = \frac{Z}{\sqrt{N}}$) ou delta de Cliff para o teste nao parametrico.

## 7. Pergunta 4 - Gasto total e planeta de origem

**Pergunta de negocio:** o gasto total a bordo difere entre os passageiros originarios de Earth, Europa e Mars?

- **Variaveis:** `TotalSpend` (quantitativa) e `HomePlanet` (categorica com 3 niveis: Earth, Europa, Mars).
- **Unidade de analise:** um passageiro.
- **Objetivo analitico:** verificar se a origem do passageiro influencia o volume financeiro gasto na viagem espacial. Para direcionamento estrategico, o parametro central de decisao e a media de gastos por planeta ($\mu_{\text{Earth}}, \mu_{\text{Europa}}, \mu_{\text{Mars}}$), apoiada pela analise nao parametrica de postos.

**Hierarquia de testes e hipoteses pre-registradas:**

1. **Teste Global Principal (orientador da conclusao de negocio): ANOVA de Welch (One-Way Welch ANOVA)**
   - **Foco:** igualdade simultanea das medias populacionais entre os 3 planetas.
   - **$H_0$ (nula):** $\mu_{\text{Earth}} = \mu_{\text{Europa}} = \mu_{\text{Mars}}$ (as medias populacionais de gasto total sao identicas entre os tres planetas de origem).
   - **$H_1$ (alternativa):** pelo menos uma media populacional difere das demais ($\exists\, i \neq j \text{ tal que } \mu_i \neq \mu_j$).
   - **Justificativa metodologica:** amostras expressivas ($n_{\text{Earth}} = 4.715$, $n_{\text{Europa}} = 2.174$, $n_{\text{Mars}} = 1.804$) garantem estabilidade amostral via TCL. A ANOVA de Welch (`scipy.stats` ou `pingouin.welch_anova`) relaxa o pressuposto de igualdade de variancias (homocedasticidade), sendo a escolha padrao diante da esperada heterocedasticidade verificada pelo Teste de Levene.

2. **Teste Global Complementar (robustez de postos): Kruskal-Wallis (H)**
   - **Foco:** igualdade das distribuicoes de postos entre os 3 grupos independentes.
   - **$H_0$ (nula):** as distribuicoes de postos de `TotalSpend` sao identicas entre Earth, Europa e Mars.
   - **$H_1$ (alternativa):** pelo menos um planeta apresenta dominancia estocastica nos postos de gasto em relacao aos demais.
   - **Justificativa metodologica:** avalia se a heterogeneidade persiste em uma escala puramente ordinal e livre de distribuicao, minimizando a influencia de passageiros com consumos atipicos.

**Protocolo pre-estabelecido de comparacoes multiplas pos-hoc (Post-Hoc):**

Se a hipotese nula global for rejeitada no teste principal ($p < 0{,}05$), sera necessario determinar exatamente quais pares de planetas apresentam divergencia estatistica:
- **Metodo Principal Pos-Hoc:** **Teste de Games-Howell** para comparacoes pareadas (Earth vs Europa, Earth vs Mars, Europa vs Mars).
  - *Criterio:* o Games-Howell e o teste pos-hoc indicado para ANOVA de Welch, pois nao assume variancias iguais nem tamanhos amostrais identicos, controlando rigorosamente a taxa de erro tipo I por familia (*Family-Wise Error Rate - FWER*) em $\alpha = 0{,}05$.
- **Metodo Complementar Pos-Hoc:** **Teste de Dunn** com correcao de **Bonferroni** (ou **Holm-Bonferroni**) sobre os postos caso o aprofundamento decorra do Kruskal-Wallis.
- **Tamanho do Efeito da ANOVA (na KAN-27):** medido por Eta ao Quadrado ($\eta^2 = \frac{SS_{\text{between}}}{SS_{\text{total}}}$) ou Omega ao Quadrado ($\omega^2$) para quantificar a proporcao da variancia de gastos explicada pelo planeta de origem.

## 8. Limite desta entrega e proximos passos

A KAN-24 estabelece o compromisso metodologico: perguntas, hipoteses, testes principais/complementares e protocolos pos-hoc foram definidos antes de consultar resultados inferenciais.

| Proxima tarefa | Entrega esperada |
| --- | --- |
| KAN-25 | Tabelas de contingencia e testes qui-quadrado para as perguntas 1 e 2 |
| KAN-26 | Levene, Welch t (principal), Mann-Whitney (complementar), IC 95% e tamanhos de efeito para a pergunta 3 |
| KAN-27 | Levene, ANOVA de Welch (principal), Kruskal-Wallis (complementar), Games-Howell pos-hoc e tamanhos de efeito para a pergunta 4 |
| KAN-28 | Tabela-resumo consolidada de todos os testes |

Portanto, este notebook para antes de calcular qualquer estatistica inferencial. Essa separacao mantem a analise transparente e protege a interpretacao dos resultados posteriores.